# Hafta 13 - SMS Spam Tespiti

Bu defterde metin sınıflandırma yöntemleriyle SMS spam tespiti yapacağız.

## İçerik
1. Veri Seti Oluşturma
2. Metin Ön İşleme
3. TF-IDF Vektörizasyonu
4. Model Eğitimi (LogisticRegression, Naive Bayes)
5. Model Karşılaştırması
6. Yeni Mesajlarla Test
7. Pipeline Oluşturma

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline

print("Kütüphaneler başarıyla yüklendi!")

## 1. Gerçek UCI SMS Spam Collection Verisini Yükleme

**SMS Spam Collection Dataset** — 5.574 gerçek SMS mesajı (4.827 normal + 747 spam). UCI Machine Learning Repository'den alınmış referans veri setidir.

**Kaynak:** [Kaggle - SMS Spam Collection](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [ ]:
# UCI SMS Spam Collection Dataset (gerçek veri)
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Label'ı sayıya çevir
df['label_num'] = (df['label'] == 'spam').astype(int)

print(f"Veri seti boyutu: {df.shape}")
print(f"\nSınıf dağılımı:")
print(df['label'].value_counts())
print(f"\nSpam oranı: {df['label_num'].mean():.2%}")

# Mesaj uzunluğu analizi
df['mesaj_uzunlugu'] = df['message'].apply(len)
print(f"\nOrtalama mesaj uzunluğu:")
print(df.groupby('label')['mesaj_uzunlugu'].mean().round(0))

# Örnek mesajlar
print("\n--- Normal (Ham) Mesaj Örnekleri ---")
for msg in df[df['label'] == 'ham']['message'].sample(3, random_state=42).values:
    print(f"  • {msg[:100]}...")
    
print("\n--- Spam Mesaj Örnekleri ---")
for msg in df[df['label'] == 'spam']['message'].sample(3, random_state=42).values:
    print(f"  • {msg[:100]}...")

df.head()

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Veri dağılımı görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Etiket dağılımı
renkler = ['#2ecc71', '#e74c3c']
df['etiket'].value_counts().plot.bar(ax=axes[0], color=renkler, edgecolor='black')
axes[0].set_title('Mesaj Türü Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tür')
axes[0].set_ylabel('Sayı')
axes[0].set_xticklabels(['Normal (Ham)', 'Spam'], rotation=0)

# Mesaj uzunluğu dağılımı
df['uzunluk'] = df['mesaj'].str.len()
df[df['etiket'] == 'ham']['uzunluk'].hist(ax=axes[1], bins=30, alpha=0.6, label='Normal', color='#2ecc71', edgecolor='black')
df[df['etiket'] == 'spam']['uzunluk'].hist(ax=axes[1], bins=30, alpha=0.6, label='Spam', color='#e74c3c', edgecolor='black')
axes[1].set_title('Mesaj Uzunluğu Dağılımı', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Karakter Sayısı')
axes[1].set_ylabel('Frekans')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Ortalama mesaj uzunluğu - Normal: {df[df['etiket']=='ham']['uzunluk'].mean():.0f} karakter")
print(f"Ortalama mesaj uzunluğu - Spam: {df[df['etiket']=='spam']['uzunluk'].mean():.0f} karakter")

## 2. Metin Ön İşleme

In [ ]:
def metin_temizle(metin):
    """Metin ön işleme fonksiyonu."""
    # Küçük harfe dönüştür
    metin = metin.lower()
    # URL'leri kaldır
    metin = re.sub(r'http\S+|www\S+|bit\.ly\S+', 'URL', metin)
    # Telefon numaralarını kaldır
    metin = re.sub(r'0[0-9]{3}[-\s]?[0-9]{3}[-\s]?[0-9]{4}', 'TELEFON', metin)
    metin = re.sub(r'0[0-9]{2,3}-[A-Z]{3,4}', 'TELEFON', metin)
    # Emoji kaldır
    metin = re.sub(r'[^\w\s]', ' ', metin)
    # Fazla boşlukları temizle
    metin = re.sub(r'\s+', ' ', metin).strip()
    return metin

# Ön işleme uygula
df['temiz_mesaj'] = df['mesaj'].apply(metin_temizle)

# Öncesi/sonrası karşılaştırma
print("Ön İşleme Örnekleri:")
print("=" * 80)
for _, satir in df[df['etiket'] == 'spam'].head(3).iterrows():
    print(f"Önce:  {satir['mesaj']}")
    print(f"Sonra: {satir['temiz_mesaj']}")
    print("-" * 80)

## 3. TF-IDF Vektörizasyonu

### Eğitim ve Test Setlerine Ayırma

Veriyi eğitim ve test olarak ikiye bölüyoruz. `stratify` parametresi, her iki sette de sınıf dağılımının aynı kalmasını sağlar. `random_state` ile tekrarlanabilir sonuçlar elde ediyoruz.

In [ ]:
# Eğitim/test ayrımı
X_train, X_test, y_train, y_test = train_test_split(
    df['temiz_mesaj'], df['etiket'],
    test_size=0.2, random_state=42, stratify=df['etiket']
)

print(f"Eğitim seti: {len(X_train)} mesaj")
print(f"Test seti: {len(X_test)} mesaj")
print(f"\nEğitim dağılımı:")
print(y_train.value_counts())
print(f"\nTest dağılımı:")
print(y_test.value_counts())

### Metin İşleme ve NLP

Metin verisini makine öğrenmesi algoritmalarının anlayabileceği sayısal formata dönüştürüyoruz. Tokenizasyon, vektörizasyon ve önceden eğitilmiş modeller bu sürecin parçalarıdır.

In [ ]:
# TF-IDF vektörizasyonu
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF matris boyutu (eğitim): {X_train_tfidf.shape}")
print(f"TF-IDF matris boyutu (test): {X_test_tfidf.shape}")
print(f"Kelime dağarcığı boyutu: {len(tfidf.get_feature_names_out())}")

# En önemli kelimeler
ozellik_isimleri = tfidf.get_feature_names_out()
print(f"\nÖrnek özellikler: {list(ozellik_isimleri[:10])}")

## 4. Model Eğitimi

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Model 1: Lojistik Regresyon
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

lr_tahmin = lr_model.predict(X_test_tfidf)
lr_dogruluk = accuracy_score(y_test, lr_tahmin)

print("=" * 60)
print("LOJİSTİK REGRESYON SONUÇLARI")
print("=" * 60)
print(f"Doğruluk: {lr_dogruluk*100:.2f}%")
print(f"\nSınıflandırma Raporu:")
print(classification_report(y_test, lr_tahmin, target_names=['Normal', 'Spam']))

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Model 2: Naive Bayes (MultinomialNB)
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)

nb_tahmin = nb_model.predict(X_test_tfidf)
nb_dogruluk = accuracy_score(y_test, nb_tahmin)

print("=" * 60)
print("NAIVE BAYES SONUÇLARI")
print("=" * 60)
print(f"Doğruluk: {nb_dogruluk*100:.2f}%")
print(f"\nSınıflandırma Raporu:")
print(classification_report(y_test, nb_tahmin, target_names=['Normal', 'Spam']))

## 5. Model Karşılaştırması

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Karışıklık matrisleri
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lojistik Regresyon
cm_lr = confusion_matrix(y_test, lr_tahmin)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Spam'], yticklabels=['Normal', 'Spam'])
axes[0].set_title(f'Lojistik Regresyon\n(Doğruluk: %{lr_dogruluk*100:.1f})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tahmin')
axes[0].set_ylabel('Gerçek')

# Naive Bayes
cm_nb = confusion_matrix(y_test, nb_tahmin)
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Normal', 'Spam'], yticklabels=['Normal', 'Spam'])
axes[1].set_title(f'Naive Bayes\n(Doğruluk: %{nb_dogruluk*100:.1f})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tahmin')
axes[1].set_ylabel('Gerçek')

plt.tight_layout()
plt.show()

### Model karşılaştırma tablosu

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Model karşılaştırma tablosu
print("=" * 55)
print("MODEL KARŞILAŞTIRMASI")
print("=" * 55)
print(f"{'Model':<25} {'Doğruluk':>12} {'F1 (Spam)':>12}")
print("-" * 55)

from sklearn.metrics import f1_score

lr_f1 = f1_score(y_test, lr_tahmin, pos_label='spam')
nb_f1 = f1_score(y_test, nb_tahmin, pos_label='spam')

print(f"{'Lojistik Regresyon':<25} {lr_dogruluk*100:>11.2f}% {lr_f1:>11.4f}")
print(f"{'Naive Bayes':<25} {nb_dogruluk*100:>11.2f}% {nb_f1:>11.4f}")
print("=" * 55)

en_iyi = 'Lojistik Regresyon' if lr_dogruluk >= nb_dogruluk else 'Naive Bayes'
print(f"\nEn iyi model: {en_iyi}")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Doğruluk karşılaştırması çubuk grafik
modeller = ['Lojistik Regresyon', 'Naive Bayes']
dogruluklar = [lr_dogruluk * 100, nb_dogruluk * 100]
f1_skorlari = [lr_f1 * 100, nb_f1 * 100]

x = np.arange(len(modeller))
genislik = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bar1 = ax.bar(x - genislik/2, dogruluklar, genislik, label='Doğruluk', color='steelblue', edgecolor='black')
bar2 = ax.bar(x + genislik/2, f1_skorlari, genislik, label='F1 Skoru (Spam)', color='coral', edgecolor='black')

ax.set_ylabel('Skor (%)')
ax.set_title('Model Performans Karşılaştırması', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(modeller)
ax.legend()
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis='y')

# Değerleri çubukların üstüne yaz
for bar in [bar1, bar2]:
    for rect in bar:
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., height + 1,
                f'%{height:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Yeni Mesajlarla Test

### Metin İşleme ve NLP

Metin verisini makine öğrenmesi algoritmalarının anlayabileceği sayısal formata dönüştürüyoruz. Tokenizasyon, vektörizasyon ve önceden eğitilmiş modeller bu sürecin parçalarıdır.

In [ ]:
# En iyi modeli seç
en_iyi_model = lr_model if lr_dogruluk >= nb_dogruluk else nb_model
en_iyi_isim = 'Lojistik Regresyon' if lr_dogruluk >= nb_dogruluk else 'Naive Bayes'

# Yeni mesajlar
yeni_mesajlar = [
    "Bugün akşam yemeğe gelir misin?",
    "TEBRİKLER! 50.000 TL kazandınız! Hemen arayın!",
    "Yarın saat 10'da toplantı var, unutma.",
    "Bedava iPhone! Şimdi tıkla ve kazan: fake-link.com",
    "Çocuklar okuldan döndü mü?",
    "KREDİ FIRSATI! Anında onay! 0900-XXX-XXXX",
    "Hava bugün çok sıcak olacakmış.",
    "Hesabınız ele geçirildi! Hemen şifrenizi değiştirin!",
    "Film başlamak üzere, acele et!",
    "Son şans! %99 indirim sadece bugün!"
]

# Ön işleme ve tahmin
yeni_temiz = [metin_temizle(m) for m in yeni_mesajlar]
yeni_tfidf = tfidf.transform(yeni_temiz)
yeni_tahminler = en_iyi_model.predict(yeni_tfidf)

# Olasılıklar
if hasattr(en_iyi_model, 'predict_proba'):
    yeni_olasiliklar = en_iyi_model.predict_proba(yeni_tfidf)

print(f"Kullanılan model: {en_iyi_isim}")
print("=" * 75)
print(f"{'Mesaj':<50} {'Tahmin':<8} {'Güven':>8}")
print("-" * 75)

for mesaj, tahmin, proba in zip(yeni_mesajlar, yeni_tahminler, yeni_olasiliklar):
    kisaltilmis = mesaj[:47] + '...' if len(mesaj) > 47 else mesaj
    guvence = max(proba) * 100
    ikon = '🚫' if tahmin == 'spam' else '✅'
    print(f"{kisaltilmis:<50} {tahmin:<8} %{guvence:>6.1f}")

## 7. Pipeline Oluşturma

Tüm adımları (ön işleme, vektörizasyon, sınıflandırma) tek bir pipeline'da birleştirelim.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# Özel metin temizleyici transformer
class MetinTemizleyici(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return [metin_temizle(metin) for metin in X]

# Pipeline oluştur
spam_pipeline = Pipeline([
    ('temizleyici', MetinTemizleyici()),
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('siniflandirici', LogisticRegression(max_iter=1000, random_state=42))
])

# Pipeline'ı eğit (ham veriyle)
spam_pipeline.fit(df['mesaj'][:int(len(df)*0.8)], df['etiket'][:int(len(df)*0.8)])

# Test
test_mesajlari = [
    "Merhaba, nasılsın?",
    "BÜYÜK FIRSAT! Bedava hediye kazanın!",
    "Yarın sınav var, çalışmalıyız.",
    "Hesabınız askıya alındı, hemen tıklayın!"
]

pipeline_tahminler = spam_pipeline.predict(test_mesajlari)

print("Pipeline Test Sonuçları:")
print("=" * 60)
for mesaj, tahmin in zip(test_mesajlari, pipeline_tahminler):
    durum = 'SPAM' if tahmin == 'spam' else 'Normal'
    print(f"[{durum:>6}] {mesaj}")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Spam'da en önemli kelimeler
if hasattr(spam_pipeline.named_steps['siniflandirici'], 'coef_'):
    ozellik_isimleri = spam_pipeline.named_steps['tfidf'].get_feature_names_out()
    katsayilar = spam_pipeline.named_steps['siniflandirici'].coef_[0]
    
    # En çok spam'ı gösteren kelimeler
    spam_idx = np.argsort(katsayilar)[-15:][::-1]
    normal_idx = np.argsort(katsayilar)[:15]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Spam kelimeleri
    spam_kelimeler = [ozellik_isimleri[i] for i in spam_idx]
    spam_katsayilar = [katsayilar[i] for i in spam_idx]
    axes[0].barh(range(15), spam_katsayilar, color='#e74c3c', edgecolor='black')
    axes[0].set_yticks(range(15))
    axes[0].set_yticklabels(spam_kelimeler)
    axes[0].set_title('Spam İşaret Eden Kelimeler', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Katsayı (Ağırlık)')
    axes[0].invert_yaxis()
    
    # Normal kelimeleri
    normal_kelimeler = [ozellik_isimleri[i] for i in normal_idx]
    normal_katsayilar = [abs(katsayilar[i]) for i in normal_idx]
    axes[1].barh(range(15), normal_katsayilar, color='#2ecc71', edgecolor='black')
    axes[1].set_yticks(range(15))
    axes[1].set_yticklabels(normal_kelimeler)
    axes[1].set_title('Normal Mesaj İşaret Eden Kelimeler', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Katsayı (Ağırlık, mutlak)')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()

## Özet

Bu defterde öğrendiklerimiz:

1. **Veri Seti Oluşturma**: SMS benzeri mesajlardan eğitim verisi hazırlama
2. **Metin Ön İşleme**: URL, telefon numarası, noktalama temizleme
3. **TF-IDF**: Metin verisini sayısal vektörlere dönüştürme
4. **Lojistik Regresyon**: Doğrusal sınıflandırma modeli
5. **Naive Bayes**: Olasılıksal sınıflandırma modeli
6. **Pipeline**: Tüm adımları tek bir yapıda birleştirme

### Spam Tespitinde Önemli Özellikler
- Büyük harfle yazılmış kelimeler (TEBRİKLER, ACELE EDİN)
- URL ve telefon numarası içeren mesajlar
- Para miktarı ve indirim oranları
- Aciliyet ifadeleri (hemen, son şans, acele)

### Sonraki Adımlar
- Daha büyük ve gerçek veri setleriyle çalışma
- Derin öğrenme modelleri (LSTM, BERT)
- Gerçek zamanlı spam filtreleme sistemi